# city-macro-data Python 使用教程

这个 notebook 演示如何使用 `city-macro-data` Python package：安装、读取、校验、分析与更新流程。

## 1) 环境准备

如果你在项目根目录，建议先安装包：

```bash
python -m pip install .\\python-pkg
```

如果你只是使用者（不是开发者），可以安装发布后的 wheel 或私有源包。

In [15]:
!pip install city-macro-data

In [16]:
from city_macro_data import load_data, get_metadata, data_version, validate_data

## 2) 读取数据与元数据

In [17]:
df = load_data()
meta = get_metadata()

print("数据维度:", df.shape)
print("数据版本:", data_version())
print("元数据字段:", list(meta.keys()))
df.head()

数据维度: (7500, 27)
数据版本: 2026.05
元数据字段: ['data_version', 'build_time_utc', 'source_file', 'row_count', 'column_count', 'columns']


,年份,城市,省份,省份代码,城市代码,地区生产总值(万元),常住人口\r\n(万人),户籍人口数\r\n(万人),行政区域土地面积\r\n(平方公里),进出口总额（百万人民币）,...,第三产业增加值占GDP比重(%),普通高等学校在校学生数(万人),互联网宽带接入用户(万人）),年末金融机构各项贷款余额(万元),年末金融机构存款余额(万元),一般公共预算支出_教育\r\n(万元),一般公共预算\r\n支出_科学技术(万元),一般公共预算支出\r\n(万元),一般公共\r\n预算收入\r\n(万元),移动电话年末用户数\r\n(万户)
0,2000,北京市,北京市,110000,110000,32779000.0,1363.6,1113.5263,16808,408956.323199,...,0.663504,28.03,NaN,86857258.0,137457223.0,600700.0,270741.8265,4430000.0,3450000.0,461.0
1,2001,北京市,北京市,110000,110000,38616000.0,1385.1,1127.8930,16800,426249.655802,...,0.687176,33.65,1090.0000,96851287.0,156924699.0,722600.0,315404.8948,5591100.0,4541700.0,629.0
2,2002,北京市,北京市,110000,110000,45257000.0,1423.2,1142.8302,16800,434581.014100,...,0.708885,39.57,3229.3270,107998081.0,179218779.0,858200.0,368295.1784,6283500.0,5339900.0,920.0
3,2003,北京市,北京市,110000,110000,52672000.0,1456.4,1154.0633,16800,566975.904532,...,0.707530,45.45,3980.0000,120430934.0,204759796.0,988200.0,431075.8457,7348000.0,5925400.0,1109.0
4,2004,北京市,北京市,110000,110000,62525000.0,1492.7,1167.7592,16800,782784.359463,...,0.702663,49.95,3134.3425,135774452.0,237813488.0,1213900.0,505774.7624,8982800.0,7444900.0,1336.0


## 3) 数据质量校验

`validate_data()` 默认检查：
- 数据非空
- 必需列存在（默认 `年份`、`城市`）

In [18]:
validate_data()

True

In [19]:
# 自定义必需列
validate_data(["年份", "城市", "省份"])

True

## 4) 常见分析示例

In [20]:
# 例1：按年份统计城市数量
city_count_by_year = df.groupby("年份")["城市"].nunique()
city_count_by_year.head()

年份
2000    300
2001    300
2002    300
2003    300
2004    300
Name: 城市, dtype: int64

In [21]:
# 例2：查看缺失值最多的前10列
missing_top10 = df.isna().sum().sort_values(ascending=False).head(10)
missing_top10

互联网宽带接入用户(万人）)           300
常住人口\r\n(万人)              75
普通高等学校在校学生数(万人)           75
第二产业增加值(万元)               50
第一产业增加值占GDP比重(%)          50
一般公共\r\n预算收入\r\n(万元)      50
一般公共预算支出\r\n(万元)          50
一般公共预算\r\n支出_科学技术(万元)     50
地区生产总值(万元)                50
户籍人口数\r\n(万人)             50
dtype: int64

## 5) 按“城市 + 指标”提取数据（实战）

下面以“北京市的常住人口”为例。先确认你数据里城市列和指标列的精确名字。

In [22]:
# 看看所有列名，找到你要的指标字段
for c in df.columns:
    print(c)

年份
城市
省份
省份代码
城市代码
地区生产总值(万元)
常住人口
(万人)
户籍人口数
(万人)
行政区域土地面积
(平方公里)
进出口总额（百万人民币）
出口总额(百万人民币)
进口总额(百万人民币)
第一产业增加值(万元)
第二产业增加值(万元)
第三产业增加值(万元)
第一产业增加值占GDP比重(%)
第二产业增加值占GDP比重(%)
第三产业增加值占GDP比重(%)
普通高等学校在校学生数(万人)
互联网宽带接入用户(万人）)
年末金融机构各项贷款余额(万元)
年末金融机构存款余额(万元)
一般公共预算支出_教育
(万元)
一般公共预算
支出_科学技术(万元)
一般公共预算支出
(万元)
一般公共
预算收入
(万元)
移动电话年末用户数
(万户)


In [23]:
# 先定位“常住人口”字段的精确列名
# （原始列名里可能包含换行符）
candidate_cols = [c for c in df.columns if "常住人口" in str(c)]
print("匹配到的列：")
for c in candidate_cols:
    print("-", repr(c))

匹配到的列：
- '常住人口\r\n(万人)'


In [24]:
# 提取北京市“常住人口”时间序列
city_name = "北京市"
indicator_col = candidate_cols[0]  # 通常类似 "常住人口\r\n(万人)"

bj_pop = (
    df.loc[df["城市"] == city_name, ["年份", indicator_col]]
      .rename(columns={indicator_col: "常住人口(万人)"})
      .sort_values("年份")
      .reset_index(drop=True)
)

bj_pop.head()

,年份,常住人口(万人)
0,2000,1363.6
1,2001,1385.1
2,2002,1423.2
3,2003,1456.4
4,2004,1492.7


In [25]:
# 通用函数：提取任意“城市 + 指标”序列
def get_city_indicator(df, city, indicator_keyword, year_col="年份", city_col="城市"):
    matches = [c for c in df.columns if indicator_keyword in str(c)]
    if not matches:
        raise ValueError(f"没有任何列包含关键字 {indicator_keyword!r}")
    indicator_col = matches[0]

    out = (
        df.loc[df[city_col] == city, [year_col, indicator_col]]
          .rename(columns={indicator_col: indicator_keyword})
          .dropna(subset=[indicator_keyword])
          .sort_values(year_col)
          .reset_index(drop=True)
    )
    return out

get_city_indicator(df, "北京市", "常住人口").head()

,年份,常住人口
0,2000,1363.6
1,2001,1385.1
2,2002,1423.2
3,2003,1456.4
4,2004,1492.7


In [26]:
# 比较同一指标在多个城市间的变化
cities = ["北京市", "上海市", "广州市", "深圳市"]
series_list = []
for c in cities:
    s = get_city_indicator(df, c, "常住人口").rename(columns={"常住人口": c})
    series_list.append(s.set_index("年份"))

comparison = series_list[0]
for s in series_list[1:]:
    comparison = comparison.join(s, how="outer")

comparison.head()

,北京市,上海市,广州市,深圳市
年份,,,,
2000,1363.6,1608.63,994.80,701.24
2001,1385.1,1668.33,996.75,724.57
2002,1423.2,1712.97,984.76,746.62
2003,1456.4,1765.84,972.93,778.27
2004,1492.7,1834.98,966.06,800.80


提示：如果运行时报 `KeyError: '常住人口'`，说明你的列名和示例不一致。请先运行“打印列名”单元，复制真实列名替换 `indicator_col`。